# Uchambuzi wa Dai za Matumizi

Daftari hii inaonyesha jinsi ya kuunda maajenti wanaotumia viendelezi kushughulikia matumizi ya safari kutoka kwa picha za risiti za ndani, kuunda barua pepe ya dai la matumizi, na kuonyesha data za matumizi kwa kutumia chati ya pai. Maajenti huchagua kazi kulingana na muktadha wa kazi kwa njia ya mabadiliko.

Hatua:
1. Maajenti wa OCR hushughulikia picha ya risiti ya ndani na kutoa data za matumizi ya safari.
2. Maajenti wa Barua pepe hutoa barua pepe ya dai la matumizi.

### Mfano wa hali ya matumizi ya safari:
Fikiria wewe ni mfanyakazi anayesafiri kwenda mkutano wa kibiashara katika mji mwingine. Kampuni yako ina sera ya kulipa tena gharama zote zinazowezekana zinazohusiana na safari. Hapa kuna muhtasari wa matumizi yanayoweza kutokea:
- Usafiri:
Gharama ya tiketi ya ndege ya safari ya mzunguko kutoka mji wako wa makazi hadi mji wa kitalabu.
Taksi au huduma za usafiri wa kubeba kutoka na kwenda uwanja wa ndege.
Usafiri wa ndani katika mji wa kitalabu (kama usafiri wa umma, magari ya kodi, au taksi).

- Malazi:
Kukaa hoteli kwa usiku tatu katika hoteli ya wastani ya biashara karibu na mahali pa mkutano.

- Chakula:
Kiwango cha kila siku cha chakula cha kiamsha kinywa, chakula cha mchana, na chakula cha jioni, kulingana na sera ya kampuni ya kwa siku.

- Matumizi Mengine:
Ada za kuegesha magari uwanja wa ndege.
Ada za upatikanaji wa intaneti kwenye hoteli.
Tipu au ada ndogo za huduma.

- Nyaraka:
Unawasilisha risiti zote (ndege, taksi, hoteli, chakula, nk) na ripoti kamili ya matumizi kwa ajili ya kurudishiwa gharama.


## Ingiza maktaba zinazohitajika

Ingiza maktaba na moduli zinazohitajika kwa daftari.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Fafanua Mifano ya Gharama

 Unda mfano wa Pydantic kwa ajili ya gharama binafsi na darasa la ExpenseFormatter kubadilisha swali la mtumiaji kuwa data ya gharama yenye muundo.

 Kila gharama itawakilishwa kwa muundo wa:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Kueleza Vifaa - Kutengeneza Barua Pepe

Tengeneza kifaa cha kazi ya kutengeneza barua pepe ya kuwasilisha dai la gharama.
- Kifaa hiki kinatumia dekoreta ya `@tool` kutoka kwa Microsoft Agent Framework.
- Kinahesabu jumla ya gharama na kuunda maelezo kuwa mwili wa barua pepe.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Chombo cha Kuchukua Gharama za Safari kutoka kwa Picha za Risiti

Tengeneza kifaa chenye kazi ya kuchukua gharama za safari kutoka kwa picha za risiti.
- Kifaa hiki kinatumia dekoreta `@tool` kutoka kwenye Microsoft Agent Framework.
- Kina soma picha ya risiti, kuiweka katika muundo wa base64, na kurudisha data URI kwa wakala kuchambua.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Matumizi ya Usindikaji

Tafsiri mawakala na waiunganishe katika mtiririko wa kazi wa mfululizo ukitumia `WorkflowBuilder`.
- Wakala wa OCR huchota data ya matumizi iliyopangwa kutoka kwenye picha ya risiti kwa kutumia chombo cha `load_receipt_image`.
- Wakala wa Barua Pepe huchukua data iliyochotwa na kuunda barua pepe ya dai la matumizi ya kitaalamu kwa kutumia chombo cha `generate_expense_email`.
- `WorkflowBuilder` na `add_edge` huunda mnyororo wa mfululizo: Wakala wa OCR → Wakala wa Barua Pepe.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Kazi kuu

Tengeneza mtiririko wa kazi wa mfululizo na uendeshe kushughulikia picha ya risiti na kuunda barua pepe ya dai la gharama.


> **Kumbuka:** Mchakato huu wa kazi kwa sasa hupitisha picha ya risiti kama maandishi ya base64, ambayo mifano mingi ya mazungumzo (ikiwa ni pamoja na gpt-5-mini) haitatambua kama picha.
> Inaweza pia kuzidi dirisha la muktadha la mfano. Inapendekezwa kutumia OCR kwa Azure AI Vision (au njia nyingine ya OCR) na kupitisha maandishi yaliyotolewa tu, au kubadilisha ili kutuma picha kama ujumbe wa `image_url`.
> Ikiwa unataka tu kuepuka makosa ya muktadha, jaribu picha ndogo ya risiti au mfano wenye dirisha kubwa la muktadha.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Kionyozo**:
Hati hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI [Co-op Translator](https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kupata usahihi, tafadhali fahamu kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au upungufu wa usahihi. Hati ya asili katika lugha yake halisi inapaswa kuchukuliwa kama chanzo cha mamlaka. Kwa taarifa muhimu, tafsiri ya kitaalamu inayofanywa na binadamu inapendekezwa. Hatutojibu kwa kuelewa vibaya au tafsiri potofu zinazotokea kutokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
